In [1]:
import numpy as np
import pandas as pd
import pvlib
from shapely.geometry import Polygon

In [2]:
def get_solar_tracking_data(latitude, longitude, timestamp_str):
    times = pd.date_range(timestamp_str, periods=1, freq='min', tz='Asia/Kolkata')
    solpos = pvlib.solarposition.get_solarposition(times, latitude, longitude)

    zenith = solpos['apparent_zenith'].iloc[0]
    azimuth = solpos['azimuth'].iloc[0]
    elevation = solpos['elevation'].iloc[0]

    zenith_rad = np.radians(zenith)
    azimuth_rad = np.radians(azimuth)

    # Unit vector pointing towards the sun (ENU coordinates)
    S_x = np.sin(zenith_rad) * np.sin(azimuth_rad)
    S_y = np.sin(zenith_rad) * np.cos(azimuth_rad)
    S_z = np.cos(zenith_rad)
    sun_vec = np.array([S_x, S_y, S_z])

    # Optimal surface angles for direct solar exposure
    optimal_tilt = zenith
    optimal_azimuth = azimuth

    return sun_vec, zenith, azimuth, elevation, optimal_tilt, optimal_azimuth

In [3]:
def calculate_dynamic_shading(panel_origin, panel_width, panel_height, panel_tilt, panel_azimuth, obstacle_vertices, sun_vec):
    tilt_r = np.radians(panel_tilt)
    azim_r = np.radians(panel_azimuth)

    # Panel normal vector
    N = np.array([
        np.sin(tilt_r) * np.sin(azim_r),
        np.sin(tilt_r) * np.cos(azim_r),
        np.cos(tilt_r)
    ])

    d = -np.array(sun_vec)

    if np.abs(np.dot(N, d)) < 1e-6:
        return 0.0, 0.0, panel_width * panel_height, "Sun ray parallel to panel"

    # Project 3D obstacle points onto panel plane
    projected_3d_points = []
    for v_obs in obstacle_vertices:
        t = np.dot(N, (panel_origin - v_obs)) / np.dot(N, d)
        if t > 0:
            P_int = v_obs + t * d
            projected_3d_points.append(P_int)

    total_panel_area = panel_width * panel_height

    if len(projected_3d_points) < 3:
        return 0.0, 0.0, total_panel_area, "No shadow on panel plane"

    # Local 2D basis (U, V) on panel surface
    u_axis = np.array([np.cos(azim_r), -np.sin(azim_r), 0])
    v_axis = np.cross(N, u_axis)

    shadow_2d_points = []
    for P in projected_3d_points:
        vec_from_origin = P - panel_origin
        u_coord = np.dot(vec_from_origin, u_axis)
        v_coord = np.dot(vec_from_origin, v_axis)
        shadow_2d_points.append((u_coord, v_coord))

    panel_poly = Polygon([
        (0, 0),
        (panel_width, 0),
        (panel_width, panel_height),
        (0, panel_height)
    ])

    shadow_poly = Polygon(shadow_2d_points).convex_hull

    if not panel_poly.intersects(shadow_poly):
        return 0.0, 0.0, total_panel_area, "Shadow outside panel"

    shaded_area_m2 = panel_poly.intersection(shadow_poly).area
    unshaded_area_m2 = total_panel_area - shaded_area_m2
    shade_pct = (shaded_area_m2 / total_panel_area) * 100.0

    return shade_pct, shaded_area_m2, unshaded_area_m2, "Shading calculated"

In [4]:
# Location & Date Setup (Pune, India)
lat, lon = 18.5204, 73.8567
date_str = '2026-06-21'

# Panel Dimensions (1m wide, 2m tall, fixed base angle: South facing, 30° tilt)
panel_origin = np.array([0.0, 0.0, 1.0])
panel_w, panel_h = 1.0, 2.0
current_tilt = 30.0
current_azimuth = 180.0

# 3D Obstacle Vertices
obstacle_vertices = [
    np.array([0.5, 1.2, 2.2]),
    np.array([1.5, 1.2, 2.2]),
    np.array([1.5, 1.8, 2.2]),
    np.array([0.5, 1.8, 2.2])
]

# Generate hourly daylight timesteps
time_range = pd.date_range(start=f"{date_str} 06:00:00", end=f"{date_str} 18:00:00", freq='1h', tz='Asia/Kolkata')

In [5]:
print(f"=== DYNAMIC SOLAR TRACKING & SHADOW ANALYSIS ===")
print(f"Location: Pune ({lat}, {lon}) | Date: {date_str}\n")

for current_time in time_range:
    time_label = current_time.strftime("%I:%M %p")
    sun_vec, zenith, azimuth, elevation, opt_tilt, opt_azimuth = get_solar_tracking_data(
        lat, lon, current_time.strftime('%Y-%m-%d %H:%M:%S')
    )

    if elevation < 0:
        continue

    shade_pct, shaded_m2, unshaded_m2, status = calculate_dynamic_shading(
        panel_origin, panel_w, panel_h, current_tilt, current_azimuth, obstacle_vertices, sun_vec
    )

    # Angular movements required to face the sun directly
    move_tilt_delta = opt_tilt - current_tilt
    move_azimuth_delta = opt_azimuth - current_azimuth

    print(f"--- Time: {time_label} ---")
    print(f"  Solar Zenith: {zenith:.1f}° | Solar Azimuth: {azimuth:.1f}°")
    print(f"  Optimal Panel Position : Tilt = {opt_tilt:.1f}°, Azimuth = {opt_azimuth:.1f}°")
    print(f"  Panel Adjustment Needed: Move Tilt by {move_tilt_delta:+.1f}°, Move Azimuth by {move_azimuth_delta:+.1f}°")
    print(f"  Shadow Area Consumed   : {shaded_m2:.3f} m² ({shade_pct:.1f}% coverage)")
    print(f"  Active Unshaded Area   : {unshaded_m2:.3f} m² / {panel_w * panel_h:.1f} m²")
    print(f"  Status                 : {status}\n")

=== DYNAMIC SOLAR TRACKING & SHADOW ANALYSIS ===
Location: Pune (18.5204, 73.8567) | Date: 2026-06-21

--- Time: 07:00 AM ---
  Solar Zenith: 77.4° | Solar Azimuth: 69.2°
  Optimal Panel Position : Tilt = 77.4°, Azimuth = 69.2°
  Panel Adjustment Needed: Move Tilt by +47.4°, Move Azimuth by -110.8°
  Shadow Area Consumed   : 0.000 m² (0.0% coverage)
  Active Unshaded Area   : 2.000 m² / 2.0 m²
  Status                 : Shadow outside panel

--- Time: 08:00 AM ---
  Solar Zenith: 64.1° | Solar Azimuth: 72.3°
  Optimal Panel Position : Tilt = 64.1°, Azimuth = 72.3°
  Panel Adjustment Needed: Move Tilt by +34.1°, Move Azimuth by -107.7°
  Shadow Area Consumed   : 0.000 m² (0.0% coverage)
  Active Unshaded Area   : 2.000 m² / 2.0 m²
  Status                 : Shadow outside panel

--- Time: 09:00 AM ---
  Solar Zenith: 50.4° | Solar Azimuth: 74.5°
  Optimal Panel Position : Tilt = 50.4°, Azimuth = 74.5°
  Panel Adjustment Needed: Move Tilt by +20.4°, Move Azimuth by -105.5°
  Shadow Area 